## 1. Setup and Data Loading

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("mini-project-train-model")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

Spark version: 3.5.3
Master: spark://spark-master:7077


In [2]:
# Load the full prepared dataset
full_df = spark.read.parquet("../outputs/exported_dataset")

print("Total rows:", full_df.count())
full_df.printSchema()
full_df.show(5, truncate=False)

Total rows: 2857
root
 |-- season: string (nullable = true)
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: long (nullable = true)
 |-- away_goals: long (nullable = true)
 |-- total_goals: long (nullable = true)
 |-- goal_difference: long (nullable = true)
 |-- result: string (nullable = true)
 |-- match_year: integer (nullable = true)

+------------------------+----------+---------------+-----------+----------+----------+-----------+---------------+--------+----------+
|season                  |date      |home_team      |away_team  |home_goals|away_goals|total_goals|goal_difference|result  |match_year|
+------------------------+----------+---------------+-----------+----------+----------+-----------+---------------+--------+----------+
|Championnat D1 2007/2008|2008-08-02|ASAC Concorde  |Nasr Sebkha|2         |0         |2          |2              |home_win|2008      |
|Championnat D1 2007/2008|2008-0

## 2. Train / Test Split

- **Training set**: All historical seasons (before 2024/2025)
- **Test set**: Scraped 2024/2025 season results

This reflects the real-world scenario where we train on past data and predict future match outcomes.

In [3]:
# Split: historical = training, 2024/2025 = test
train_raw = full_df.filter(F.col("season") != "2024/2025")
test_raw = full_df.filter(F.col("season") == "2024/2025")

print(f"Training set: {train_raw.count()} rows")
print(f"Test set:     {test_raw.count()} rows")

print("\nResult distribution (train):")
train_raw.groupBy("result").count().orderBy("result").show()

print("Result distribution (test):")
test_raw.groupBy("result").count().orderBy("result").show()

Training set: 2590 rows
Test set:     267 rows

Result distribution (train):
+--------+-----+
|  result|count|
+--------+-----+
|away_win|  859|
|    draw|  707|
|home_win| 1024|
+--------+-----+

Result distribution (test):
+--------+-----+
|  result|count|
+--------+-----+
|away_win|   72|
|    draw|   81|
|home_win|  114|
+--------+-----+



## 3. Feature Engineering

We compute **team-level historical aggregates** from the training data:

| Feature | Description |
|---------|-------------|
| `home_avg_scored` | Avg goals scored by home team (when playing at home) |
| `home_avg_conceded` | Avg goals conceded by home team (at home) |
| `home_win_rate` | Win rate of home team (at home) |
| `away_avg_scored` | Avg goals scored by away team (when playing away) |
| `away_avg_conceded` | Avg goals conceded by away team (away) |
| `away_win_rate` | Win rate of away team (when playing away) |

These features capture each team's historical strength profile.

In [4]:
# --- Home team stats (from training data only) ---
home_stats = (
    train_raw
    .groupBy("home_team")
    .agg(
        F.avg("home_goals").alias("home_avg_scored"),
        F.avg("away_goals").alias("home_avg_conceded"),
        F.avg(F.when(F.col("result") == "home_win", 1.0).otherwise(0.0)).alias("home_win_rate"),
        F.count("*").alias("home_matches"),
    )
)

print("Home stats (sample):")
home_stats.orderBy(F.desc("home_matches")).show(10, truncate=False)

Home stats (sample):
+-----------------+------------------+------------------+-------------------+------------+
|home_team        |home_avg_scored   |home_avg_conceded |home_win_rate      |home_matches|
+-----------------+------------------+------------------+-------------------+------------+
|Ksar             |1.236180904522613 |1.185929648241206 |0.3768844221105528 |199         |
|Tevragh-Zeina    |1.8080808080808082|0.6262626262626263|0.6111111111111112 |198         |
|Nouadhibou       |1.7766497461928934|0.5279187817258884|0.6395939086294417 |197         |
|SNIM             |1.3020833333333333|0.8854166666666666|0.4270833333333333 |192         |
|ASAC Concorde    |1.7277486910994764|0.8952879581151832|0.5549738219895288 |191         |
|Tidjikja         |1.382716049382716 |1.0555555555555556|0.43209876543209874|162         |
|Police           |0.9054054054054054|1.3716216216216217|0.24324324324324326|148         |
|Garde Nationale  |1.027972027972028 |1.2447552447552448|0.3076923076

In [5]:
# --- Away team stats (from training data only) ---
away_stats = (
    train_raw
    .groupBy("away_team")
    .agg(
        F.avg("away_goals").alias("away_avg_scored"),
        F.avg("home_goals").alias("away_avg_conceded"),
        F.avg(F.when(F.col("result") == "away_win", 1.0).otherwise(0.0)).alias("away_win_rate"),
        F.count("*").alias("away_matches"),
    )
)

print("Away stats (sample):")
away_stats.orderBy(F.desc("away_matches")).show(10, truncate=False)

Away stats (sample):
+-----------------+------------------+------------------+-------------------+------------+
|away_team        |away_avg_scored   |away_avg_conceded |away_win_rate      |away_matches|
+-----------------+------------------+------------------+-------------------+------------+
|Nouadhibou       |1.5555555555555556|0.7323232323232324|0.5757575757575758 |198         |
|Tevragh-Zeina    |1.4111675126903553|0.8324873096446701|0.5076142131979695 |197         |
|Ksar             |1.0765306122448979|1.1173469387755102|0.32142857142857145|196         |
|SNIM             |1.0205128205128204|1.0153846153846153|0.3384615384615385 |195         |
|ASAC Concorde    |1.4404145077720207|0.9948186528497409|0.45595854922279794|193         |
|Tidjikja         |1.14375           |0.93125           |0.38125            |160         |
|Police           |0.8513513513513513|1.2972972972972974|0.2702702702702703 |148         |
|Garde Nationale  |0.9929577464788732|1.147887323943662 |0.2746478873

In [6]:
def add_features(df, home_stats_df, away_stats_df):
    """Join team-level stats onto a match DataFrame.
    Uses string-based join keys so PySpark auto-deduplicates columns."""
    # Join home stats (string join key = no duplicate columns)
    with_home = (
        df.join(home_stats_df, on="home_team", how="left")
        .drop("home_matches")
    )
    
    # Join away stats
    with_both = (
        with_home.join(away_stats_df, on="away_team", how="left")
        .drop("away_matches")
    )
    
    # Fill nulls for teams not found in historical data
    for col_name in ["home_avg_scored", "home_avg_conceded", "home_win_rate",
                     "away_avg_scored", "away_avg_conceded", "away_win_rate"]:
        with_both = with_both.fillna({col_name: 0.5})
    
    return with_both


train_featured = add_features(train_raw, home_stats, away_stats)
test_featured = add_features(test_raw, home_stats, away_stats)

print(f"Training with features: {train_featured.count()} rows")
print(f"Test with features:     {test_featured.count()} rows")

train_featured.select(
    "home_team", "away_team", "result",
    "home_avg_scored", "home_avg_conceded", "home_win_rate",
    "away_avg_scored", "away_avg_conceded", "away_win_rate"
).show(10, truncate=False)

print("\nTest set sample:")
test_featured.select(
    "home_team", "away_team", "result",
    "home_avg_scored", "home_win_rate",
    "away_avg_scored", "away_win_rate"
).show(10, truncate=False)

Training with features: 2590 rows
Test with features:     267 rows
+---------------+---------------+--------+------------------+------------------+-------------------+------------------+------------------+-------------------+
|home_team      |away_team      |result  |home_avg_scored   |home_avg_conceded |home_win_rate      |away_avg_scored   |away_avg_conceded |away_win_rate      |
+---------------+---------------+--------+------------------+------------------+-------------------+------------------+------------------+-------------------+
|ASAC Concorde  |Nasr Sebkha    |home_win|1.7277486910994764|0.8952879581151832|0.5549738219895288 |1.0606060606060606|1.0               |0.30303030303030304|
|Police         |Tidjikja       |away_win|0.9054054054054054|1.3716216216216217|0.24324324324324326|1.14375           |0.93125           |0.38125            |
|Ksar           |SNIM           |home_win|1.236180904522613 |1.185929648241206 |0.3768844221105528 |1.0205128205128204|1.0153846153846153|

## 4. ML Pipeline Setup

We use Spark MLlib's pipeline API:
1. **StringIndexer** — encode `result` (home_win/draw/away_win) as numeric labels
2. **VectorAssembler** — combine all features into a single vector column
3. **Classifier** — Random Forest or Logistic Regression

In [7]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Define the feature columns
feature_cols = [
    "home_avg_scored",
    "home_avg_conceded",
    "home_win_rate",
    "away_avg_scored",
    "away_avg_conceded",
    "away_win_rate",
]

# Stage 1: Index the target label
label_indexer = StringIndexer(
    inputCol="result",
    outputCol="label",
    handleInvalid="keep"
)

# Stage 2: Assemble features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

print("Feature columns:", feature_cols)
print("Pipeline stages: StringIndexer → VectorAssembler → Classifier")

Feature columns: ['home_avg_scored', 'home_avg_conceded', 'home_win_rate', 'away_avg_scored', 'away_avg_conceded', 'away_win_rate']
Pipeline stages: StringIndexer → VectorAssembler → Classifier


## 5. Model 1 — Random Forest Classifier

In [8]:
# Random Forest
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=5,
    seed=42,
)

rf_pipeline = Pipeline(stages=[label_indexer, assembler, rf])

print("Training Random Forest model ...")
rf_model = rf_pipeline.fit(train_featured)
print("Training complete.")

Training Random Forest model ...
Training complete.


In [9]:
# Evaluate Random Forest on test set
rf_predictions = rf_model.transform(test_featured)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

rf_accuracy = evaluator_acc.evaluate(rf_predictions)
rf_f1 = evaluator_f1.evaluate(rf_predictions)

print("=" * 50)
print("Random Forest Results")
print("=" * 50)
print(f"  Accuracy:       {rf_accuracy:.4f}")
print(f"  Weighted F1:    {rf_f1:.4f}")

Random Forest Results
  Accuracy:       0.2697
  Weighted F1:    0.1730


In [10]:
# Feature importances
rf_classifier = rf_model.stages[-1]
importances = rf_classifier.featureImportances

print("Feature Importances (Random Forest):")
print("-" * 40)
for i, col_name in enumerate(feature_cols):
    print(f"  {col_name:25s} : {importances[i]:.4f}")

Feature Importances (Random Forest):
----------------------------------------
  home_avg_scored           : 0.1706
  home_avg_conceded         : 0.1545
  home_win_rate             : 0.2358
  away_avg_scored           : 0.1246
  away_avg_conceded         : 0.1540
  away_win_rate             : 0.1606


## 6. Model 2 — Logistic Regression (Comparison)

In [11]:
# Logistic Regression
lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=100,
    family="multinomial",
)

lr_pipeline = Pipeline(stages=[label_indexer, assembler, lr])

print("Training Logistic Regression model ...")
lr_model = lr_pipeline.fit(train_featured)
print("Training complete.")

Training Logistic Regression model ...
Training complete.


In [12]:
# Evaluate Logistic Regression on test set
lr_predictions = lr_model.transform(test_featured)

lr_accuracy = evaluator_acc.evaluate(lr_predictions)
lr_f1 = evaluator_f1.evaluate(lr_predictions)

print("=" * 50)
print("Logistic Regression Results")
print("=" * 50)
print(f"  Accuracy:       {lr_accuracy:.4f}")
print(f"  Weighted F1:    {lr_f1:.4f}")

Logistic Regression Results
  Accuracy:       0.2697
  Weighted F1:    0.1677


## 7. Model Comparison

In [13]:
# Side-by-side comparison
comparison_data = [
    ("Random Forest", round(rf_accuracy, 4), round(rf_f1, 4)),
    ("Logistic Regression", round(lr_accuracy, 4), round(lr_f1, 4)),
]

comparison_df = spark.createDataFrame(comparison_data, ["Model", "Accuracy", "Weighted_F1"])
comparison_df.show(truncate=False)

best_model_name = "Random Forest" if rf_f1 >= lr_f1 else "Logistic Regression"
print(f"\n→ Best model by F1-score: {best_model_name}")

+-------------------+--------+-----------+
|Model              |Accuracy|Weighted_F1|
+-------------------+--------+-----------+
|Random Forest      |0.2697  |0.173      |
|Logistic Regression|0.2697  |0.1677     |
+-------------------+--------+-----------+


→ Best model by F1-score: Random Forest


## 8. Sample Predictions

In [14]:
# Show sample predictions from the best model
best_predictions = rf_predictions if rf_f1 >= lr_f1 else lr_predictions

print("Sample Predictions (test set — 2024/2025 season):")
print("=" * 80)

best_predictions.select(
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "result",
    "label",
    "prediction",
).show(20, truncate=False)

Sample Predictions (test set — 2024/2025 season):
+-------------------+---------------------------------+----------+----------+--------+-----+----------+
|home_team          |away_team                        |home_goals|away_goals|result  |label|prediction|
+-------------------+---------------------------------+----------+----------+--------+-----+----------+
|ASC Gendrim        |FC Inter Nouakchott     [6-5 pen]|3         |3         |draw    |2.0  |1.0       |
|FC Nouadhibou      |AS Pompiers                      |0         |1         |away_win|1.0  |1.0       |
|ASC Police         |AS Garde                         |1         |0         |home_win|0.0  |1.0       |
|King's             |FC Ittihad Nouadhibou            |5         |1         |home_win|0.0  |1.0       |
|FC Ksar            |ASC Touldé                       |2         |0         |home_win|0.0  |1.0       |
|FC Tevragh-Zeďne   |Chemal FC                        |2         |1         |home_win|0.0  |1.0       |
|AS Pompiers  

In [15]:
# Confusion-style breakdown
print("Prediction vs Actual (counts):")
best_predictions.groupBy("result", "prediction").count().orderBy("result", "prediction").show()

Prediction vs Actual (counts):
+--------+----------+-----+
|  result|prediction|count|
+--------+----------+-----+
|away_win|       0.0|    6|
|away_win|       1.0|   62|
|away_win|       2.0|    4|
|    draw|       0.0|    3|
|    draw|       1.0|   74|
|    draw|       2.0|    4|
|home_win|       0.0|    6|
|home_win|       1.0|  104|
|home_win|       2.0|    4|
+--------+----------+-----+



## 9. Save the Best Model

In [16]:
# Save the best model to disk
model_path = "../outputs/model_exports/best_model"
best_model = rf_model if rf_f1 >= lr_f1 else lr_model
best_model.write().overwrite().save(model_path)

print(f"Best model ({best_model_name}) saved to: {model_path}")

Best model (Random Forest) saved to: /workspace/data/mini_project/models/best_model


## 10. Discussion of Model Limitations

### Limitations

1. **Small dataset**: ~2590 historical matches is limited for ML; the model may not generalize well.

2. **Team name inconsistency**: Team names change across seasons (e.g., `Tevragh-Zeina` vs `FC Tevragh-Zeïne`). Some test teams may not appear in training data, resulting in default feature values.

3. **No temporal features**: The scraped 2024/2025 data has no dates, so we cannot compute recent form indicators (last 5 matches, etc.).

4. **Class imbalance**: Home wins are typically more frequent than draws or away wins, which biases the model.

5. **Static features**: Our features are season-level averages, not dynamic. A team's strength may change within a season.

6. **No external factors**: We don't account for player availability, weather, referee, or match importance.

### Possible Improvements

- **ELO rating system** for teams to capture strength evolution
- **Rolling form indicators** (last N matches performance)
- **Head-to-head statistics** between specific team pairs
- **Gradient Boosted Trees** (GBTClassifier) for potentially better performance
- **Cross-validation** with hyperparameter tuning
- **Team name standardization** using fuzzy matching across seasons

In [17]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
